# ES vs NQ index-futures statistical arbitrage — research notebook

**Goals:** test for cointegration between E-mini S&P 500 (ES) and E-mini
Nasdaq-100 (NQ) futures, estimate a hedge ratio, characterise the residual
spread as an Ornstein–Uhlenbeck process, and evaluate a z-score strategy
out of sample.

**Disclaimer:** research and education only — not financial advice, no live
trading, no order placement.

## 1. Setup & imports

Uses the installed `index_futures_stat_arb` package; falls back to adding
`../src` to `sys.path` if it is not installed. Optionally loads `.env` via
`python-dotenv` when available.

In [ ]:
import os
import sys

try:
    from dotenv import load_dotenv
    load_dotenv("../.env")
except ImportError:
    pass

try:
    import index_futures_stat_arb as ifsa
except ImportError:
    sys.path.insert(0, "../src")
    import index_futures_stat_arb as ifsa

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

## 2. Data ingestion

Default source is yfinance (`ES=F`/`NQ=F`, falling back to `SPY`/`QQQ` ETFs).
If yfinance is unavailable or returns nothing, we fall back to a synthetic
cointegrated pair so the rest of the notebook always runs.

In [ ]:
try:
    prices = ifsa.load_prices(
        source=os.environ.get("DATA_SOURCE", "yfinance"), start="2018-01-01"
    )
    print(f"Loaded {len(prices)} rows from {os.environ.get('DATA_SOURCE', 'yfinance')}")
except Exception as exc:
    print(f"Real data unavailable ({exc}); using synthetic pair.")
    prices = ifsa.generate_synthetic_pair()

prices.tail()

Optional: real CME data via Databento (requires `DATABENTO_API_KEY` and the
`databento` extra).

In [ ]:
if os.environ.get("DATABENTO_API_KEY"):
    prices = ifsa.load_prices(source="databento", start="2018-01-01")
    print(f"Loaded {len(prices)} rows from Databento")
else:
    print("DATABENTO_API_KEY not set; skipping Databento path.")

## 3. Cleaning & alignment

Sort, de-duplicate the index, invalidate non-positive prices, short forward
fill, drop remaining NaNs.

In [ ]:
prices = ifsa.align_and_clean(prices)
prices.plot(title="ES / NQ price series", subplots=True)
plt.tight_layout()
plt.show()
prices.describe()

## 4. Cointegration: Engle–Granger test

Null hypothesis: no cointegration. A low p-value supports a tradable
mean-reverting spread.

In [ ]:
logp = ifsa.log_prices(prices)
y, x = logp["ES"], logp["NQ"]

eg = ifsa.engle_granger(y, x)
print(f"statistic={eg.statistic:.3f}  p-value={eg.pvalue:.4f}  "
      f"cointegrated={eg.is_cointegrated}")
print("critical values:", eg.critical_values)

ADF test on the residual spread — stationarity check for the residuals
of the hedge regression.

In [ ]:
hedge = ifsa.estimate_hedge_ratio(y, x, method="ols")
adf = ifsa.adf_test(hedge.residuals)
print(f"ADF stat={adf['statistic']:.3f}  p-value={adf['pvalue']:.4f}  "
      f"stationary={adf['is_stationary']}")

## 5. Hedge ratio: OLS vs total least squares

In [ ]:
hedge_ols = ifsa.estimate_hedge_ratio(y, x, method="ols")
hedge_tls = ifsa.estimate_hedge_ratio(y, x, method="tls")
print(f"OLS: alpha={hedge_ols.alpha:.4f}  beta={hedge_ols.beta:.4f}  R2={hedge_ols.r_squared:.4f}")
print(f"TLS: alpha={hedge_tls.alpha:.4f}  beta={hedge_tls.beta:.4f}  R2={hedge_tls.r_squared:.4f}")

roll = ifsa.rolling_hedge_ratio(y, x, window=250)
roll["beta"].plot(title="Rolling 250d hedge ratio (beta)")
plt.axhline(hedge_ols.beta, color="k", ls="--", label="OLS beta")
plt.legend()
plt.show()

## 6. Spread and z-score

In [ ]:
spread = ifsa.compute_spread(y, x, hedge_ols.beta, hedge_ols.alpha)
z = ifsa.zscore(spread, window=60)

fig, axes = plt.subplots(2, 1, sharex=True)
spread.plot(ax=axes[0], title="Log-price spread")
z.plot(ax=axes[1], title="60d rolling z-score")
for lvl in (2.0, -2.0, 0.5, -0.5):
    axes[1].axhline(lvl, color="r" if abs(lvl) == 2.0 else "g", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Ornstein–Uhlenbeck fit

AR(1) calibration of the spread: mean-reversion speed `theta`, long-run mean
`mu`, diffusion `sigma`, and implied half-life.

In [ ]:
ou = ifsa.fit_ou(spread)
print(f"theta={ou.theta:.4f}  mu={ou.mu:.4f}  sigma={ou.sigma:.4f}")
print(f"half-life: {ou.half_life:.1f} periods")
print(f"stationary std={ou.stationary_std:.4f}  empirical std={spread.std():.4f}")

## 8. Out-of-sample walk-forward backtest

Hedge ratio and OU are fit on the train window only; the z-score strategy is
then evaluated on the held-out test window.

In [ ]:
results = ifsa.walk_forward_backtest(
    prices, split=0.7, z_window=60, entry=2.0, exit=0.5, stop=4.0
)

fig, ax = plt.subplots()
results["train"].equity.plot(ax=ax, label="train (in-sample)")
results["test"].equity.plot(ax=ax, label="test (out-of-sample)")
ax.axvline(results["test"].equity.index[0], color="k", ls=":", alpha=0.6)
ax.set_title("Spread strategy equity curve")
ax.legend()
plt.show()

pd.DataFrame(
    {"train": results["train"].metrics, "test": results["test"].metrics}
)

## 9. Caveats & next steps

- Continuous-contract roll artifacts mean the naive spread PnL is not
  exactly tradable.
- yfinance daily bars: data-quality gaps, no intraday execution modelling.
- ETF proxies (SPY/QQQ) differ from futures in tracking and financing.
- No transaction-cost, margin, or slippage model beyond a flat per-unit cost.
- Next steps: Databento intraday bars, roll-aware continuous series,
  Kalman-filter hedge ratio, proper cost/slippage model, parameter
  robustness sweeps.

*Research only — not financial advice.*